# Shreve Week 09 — Risk-Neutral Measure & Girsanov

**Week 9** — Risk-Neutral Measure & Girsanov

This notebook teaches **risk-neutral measure & girsanov** in the style of our graduate probability notebook: precise definitions from Shreve, then **verified with Python**.

## What you will learn

| Part | Topic | Shreve |
|------|-------|--------|
| 1 | **Change of measure** | Ch. 5.1 |
| 2 | **Radon-Nikodym derivative** | Ch. 5.1 |
| 3 | **Girsanov theorem** | Ch. 5.2 |
| 4 | **Risk-neutral pricing** | Ch. 5.2 |
| — | **Problem set** | Ch. 5 |

## How to use this notebook

1. **Read** each markdown cell, then **run** the code beneath it (`Shift+Enter`).
2. **Change parameters** and re-run — stochastic calculus is about *relationships*, not memorized formulas.
3. Sections end with **"Your turn"** exercises. The **problem set** at the end has **click-to-reveal solutions**.
4. Primary reference: **Shreve**, *Stochastic Calculus for Finance II* — see chapter pointers in each section.

**Shreve reference:** Ch. 5 — [Vol II PDF](https://cms.dm.uba.ar/academico/materias/2docuat2016/analisis_cuantitativo_en_finanzas/Steve_ShreveStochastic_Calculus_for_Finance_II.pdf)

Let's begin.


---
## Setup — run this first

We use NumPy for simulation, SciPy for exact distributions, and Matplotlib for plots.


In [ ]:
# If anything is missing, uncomment and run once:
# !pip install numpy scipy matplotlib

import numpy as np
import matplotlib.pyplot as plt
from scipy import stats
from scipy.special import erf

np.random.seed(42)
plt.rcParams["figure.figsize"] = (9, 5)
plt.rcParams["axes.grid"] = True

print("Ready. NumPy", np.__version__, "| SciPy", stats.__version__)


---
# Part 1 — Change of Probability Measure

Under \(P\), stock may have drift \(\mu\). Under equivalent \(Q\), discounted price is martingale: \(d(e^{-rt}S_t) = e^{-rt}\sigma S_t\, dW_t^Q\).

**Shreve Ch. 5.1:** motivation for risk-neutral measure.


In [ ]:
# Under P: E[S_T] = S_0 e^{μT}; under Q: E^Q[e^{-rT} S_T] = S_0
S0, mu, r, sigma, T = 100, 0.12, 0.05, 0.2, 1.0
rng = np.random.default_rng(90)
n = 200_000
# P measure
Z = rng.standard_normal(n)
S_T_P = S0 * np.exp((mu-0.5*sigma**2)*T + sigma*np.sqrt(T)*Z)
# Q measure: drift r instead of mu
S_T_Q = S0 * np.exp((r-0.5*sigma**2)*T + sigma*np.sqrt(T)*Z)
print(f"E[S_T] under P     = {S_T_P.mean():.2f} (S0 e^μT = {S0*np.exp(mu*T):.2f})")
print(f"E[e^{-rT}S_T] under Q = {np.exp(-r*T)*S_T_Q.mean():.2f} (S0 = {S0})")


---
# Part 2 — Radon-Nikodym Derivative

$$\frac{dQ}{dP} = Z_T = \exp\left(-\theta W_T - \frac{1}{2}\theta^2 T\right), \quad \theta = \frac{\mu - r}{\sigma}$$

\(Z_t\) is a \(P\)-martingale with \(E[Z_T] = 1\).

**Shreve Ch. 5.1:** exponential martingale as density.


In [ ]:
theta = (mu - r) / sigma
rng = np.random.default_rng(91)
Z_T = []
for _ in range(10_000):
    W_T = rng.normal(0, np.sqrt(T))
    Z_T.append(np.exp(-theta*W_T - 0.5*theta**2*T))
print(f"E[Z_T] sim = {np.mean(Z_T):.4f} (theory 1)")


---
# Part 3 — Girsanov Theorem

Under \(Q\), define \(W_t^Q = W_t + \theta t\). Then \(W^Q\) is Brownian motion under \(Q\).

Transforms drift: \(\mu S\, dt + \sigma S\, dW \Rightarrow r S\, dt + \sigma S\, dW^Q\).

**Shreve Ch. 5.2:** Girsanov theorem.


In [ ]:
# Girsanov: W^Q = W + θt has same law under Q as W under P
rng = np.random.default_rng(92)
n = 100_000
W_T = rng.normal(0, np.sqrt(T), size=n)
W_Q_T = W_T + theta * T
print(f"W_T under P:  mean={W_T.mean():.4f}, var={W_T.var():.4f}")
print(f"W^Q_T under Q: mean={W_Q_T.mean():.4f}, var={W_Q_T.var():.4f}")
print(f"θT = {theta*T:.4f}")


---
# Part 4 — Risk-Neutral Pricing

Derivative price: \(V_0 = E^Q[e^{-rT} g(S_T)]\).

Same expectation under \(Q\) gives BSM without real-world drift \(\mu\).

**Shreve Ch. 5.2:** pricing by risk-neutral expectation.


In [ ]:
K = 100
payoff = np.maximum(S_T_Q - K, 0)
price = np.exp(-r*T) * payoff.mean()
# BS reference
d1 = (np.log(S0/K)+(r+0.5*sigma**2)*T)/(sigma*np.sqrt(T))
d2 = d1 - sigma*np.sqrt(T)
bs = S0*stats.norm.cdf(d1) - K*np.exp(-r*T)*stats.norm.cdf(d2)
print(f"Risk-neutral MC = {price:.4f}, BS = {bs:.4f}")


**Your turn:** Why does option price not depend on \(\mu\)?


---
# Problem Set

**Try each problem before revealing the solution.**


## Problems

1. Define equivalent probability measures \(P\) and \(Q\).

2. Show \(Z_t = \exp(-\theta W_t - \frac{1}{2}\theta^2 t)\) is a \(P\)-martingale.

3. Apply Girsanov to transform GBM drift from \(\mu\) to \(r\).

4. Price a digital call (payoff \(1_{S_T > K}\)) under \(Q\).


<details>
<summary><strong>Reveal solutions</strong></summary>

**1.** \(Q(A)=0 \iff P(A)=0\); same null sets; \(Q(\Omega)=1\).

**2.** Itô on \(\log Z_t\) gives \(d\log Z = -\theta dW - \frac{1}{2}\theta^2 dt\), so \(dZ = -\theta Z dW\) (martingale).

**3.** \(dW^Q = dW + \theta dt\), \(\theta=(\mu-r)/\sigma\); substitute into \(dS\).

**4.** \(V_0 = e^{-rT} Q(S_T > K) = e^{-rT}\Phi(d_2)\).

</details>


---
## Further reading

- **Shreve**, *Stochastic Calculus for Finance II*, Ch. 5 — primary text for this week.
- **Shreve**, *Stochastic Calculus for Finance I* — discrete-time foundations (Ch. 1–5).
- **Karatzas & Shreve**, *Brownian Motion and Stochastic Calculus* — rigorous continuous-time theory.

Whenever a theorem says a process "converges" or a formula "holds in expectation," you can **simulate it** here and see the numbers match.
